In [1]:
from pathlib import Path
from collections import defaultdict

import scanpy as sc
import pandas as pd
import os
import sys


In [2]:
from preprocessing.scripts.data_loading import load_metadata_txt
from preprocessing.scripts.inspect_fingerprints import get_fingerprint_all, draw_molecules, analyze_fingerprint_collision_all
from preprocessing.scripts.data_preprocessing import add_fingerprints, del_false_duplicate_fp

In [3]:
METADATA_EDITED_FOLDER = Path("./Metadata_edited/")
if not os.path.exists(METADATA_EDITED_FOLDER):
    os.mkdir(METADATA_EDITED_FOLDER)

In [4]:
merged_LINCS_metadata_dir = Path("/home/ani/Documents/uni/prnet_eval/dataset/metadata/LINCS")


In [5]:
comp_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "compoundinfo_beta.txt")
gene_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "geneinfo_beta.txt")
inst_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "instinfo_beta.txt")

In [6]:
comp_info_merged = comp_info_merged[
    comp_info_merged['canonical_smiles'].notna() &
    comp_info_merged['canonical_smiles'].astype(str).str.strip().str.lower().ne('restricted')
].reset_index(drop=True)

In [7]:
comp_info_merged

,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases
0,BRD-A08715367,L-theanine,None,None,CCNC(=O)CCC(N)C(O)=O,DATAGRPVKZEWHA-UHFFFAOYSA-N,l-theanine
1,BRD-A12237696,L-citrulline,None,None,NC(CCCNC(N)=O)C(O)=O,RHGKLRLOHDJJDR-UHFFFAOYSA-N,l-citrulline
2,BRD-A18795974,BRD-A18795974,None,None,CCCN(CCC)C1CCc2ccc(O)cc2C1,BLYMJBIZMIGWFK-UHFFFAOYSA-N,7-hydroxy-DPAT
3,BRD-A27924917,BRD-A27924917,None,None,NCC(O)(CS(O)(=O)=O)c1ccc(Cl)cc1,WBSMZVIMANOCNX-UHFFFAOYSA-N,2-hydroxysaclofen
4,BRD-A35931254,BRD-A35931254,None,None,CN1CCc2cccc-3c2C1Cc1ccc(O)c(O)c-31,VMWNQDUVQKEIOC-UHFFFAOYSA-N,r(-)-apomorphine
...,...,...,...,...,...,...,...
33510,BRD-K62685538,triptorelin,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)C[C@H](NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(...,VXKHXGOKWPXYNA-PGBVPBMZSA-N,None
33511,BRD-K62221994,T-98475,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)OC(=O)c1cn(Cc2c(F)cccc2F)c3sc(c(CN(C)Cc4c...,RANJJVIMTOIWIN-UHFFFAOYSA-N,None
33512,BRD-K53397409,benzoic-acid,RAB9A,"Precursor for food preservatives, plasticizers...",OC(=O)c1ccccc1,WPYMKLBDIGXBTP-UHFFFAOYSA-N,None
33513,BRD-A62182663,YK-4279,DHX9,Binding of RNA helicase A to the transcription...,COc1ccc(cc1)C(=O)CC1(O)C(=O)Nc2c1c(Cl)ccc2Cl,HLXSCTYHLQHQDJ-UHFFFAOYSA-N,None


In [8]:
fp_methods = [
    "morgan",
    "map4",
    "map4c",
    "erg",
    "physicochemical",
    "topological_torsion",
]
smiles_list = comp_info_merged["canonical_smiles"].dropna().unique().tolist()
analysis_results = []

for method in fp_methods:
    print(f"Running analysis for: {method}")

    # Compute fingerprints for all SMILES
    bitstrings = get_fingerprint_all(smiles_list, fp_type=method)
    if bitstrings is None:
        continue

    # Map fingerprints to original SMILES
    fp_to_smiles = defaultdict(list)
    for smi, fp_str in zip(smiles_list, bitstrings):
        if fp_str is not None:
            fp_to_smiles[fp_str].append(smi)

    # Extract colliding groups
    collisions = {
        fp_str: smis for fp_str, smis in fp_to_smiles.items() if len(smis) > 1
    }
    total_collisions = len(collisions)

    # Run collision analysis function
    reason_counts = defaultdict(int)
    for group_smiles in collisions.values():
        for reason in analyze_fingerprint_collision_all(group_smiles):
            reason_counts[reason] += 1

    for reason, count in reason_counts.items():
        analysis_results.append(
            {
                "Fingerprint Method": method,
                "Collision Reason": reason,
                "Collision Count": count,
                "% Collision Groups": (
                    round((count / total_collisions) * 100, 2)
                    if total_collisions > 0
                    else 0
                ),
            }
        )

# View final pivot table comparison
df_summary = pd.DataFrame(analysis_results)
df_pivot = df_summary.pivot(
    index="Collision Reason", columns="Fingerprint Method", values="Collision Count"
).fillna(0).astype(int)

display(df_pivot)

Running analysis for: morgan


[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerator
[11:13:40] DEPRECATION WARNING: please use MorganGenerat

Running analysis for: map4
Running analysis for: map4c
Running analysis for: erg
Running analysis for: physicochemical
Running analysis for: topological_torsion


Fingerprint Method,erg,map4,map4c,morgan,physicochemical,topological_torsion
Collision Reason,,,,,,
Bioisosteres (Same Skeleton),104,0,0,162,197,16
Compositional Isomer (Different Formula),995,1,1,234,1490,13
Defined vs Undefined Stereo,248,29,9,262,252,266
Dimer/Salt Variation,1,0,0,22,35,1
Stereoisomers (R/S conflict),3313,3950,2,4000,3132,4065
Structural Isomers (Positional),152,0,0,54,77,23
True duplicates,49,49,54,70,53,73
